In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "notebooks") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

In [ ]:
# a little magic to get the output devices properly for both the rt synth and the html player
import os
os.environ["ALSA_CONFIG_PATH"] = "/usr/share/alsa/alsa.conf"
os.environ["ALSA_PLUGIN_DIR"]  = "/usr/lib/x86_64-linux-gnu/alsa-lib"

import sounddevice as sd # import *after* the os.environ settings
print("Default device:", sd.default.device)
print([d["name"] for d in sd.query_devices() if d["max_output_channels"]>0][-3:])

In [ ]:
import torch
import os
import numpy as np
from pathlib import Path
import time

import matplotlib.pyplot as plt
from IPython.display import Audio, display

from transformers import EncodecModel
from rnencodec.generator import RNNGeneratorSoft, EncodecRTPlayer

from realtime_synth_ui import build_synth_ui # pip install "rtpysynth[ui] @ git+https://github.com/lonce/RTPySynth@v0.1.4"
# # import the rt system demo synths just to have them on the interface
from realtime_synth.generators.sine import SineGenerator
from realtime_synth.generators.noisy_lp import NoisyLPGenerator

# for RNN4Control
from rnencodec.model.gru_audio_model import RNN, GRUModelConfig
from rnencodec.audioDataLoader.audio_dataset import LatentDatasetConfig
from rnencodec.audioDataLoader.audio_dataset import  efficient_codes_to_latents, preprocess_latents_for_RNN # , latents_to_audio_simple,

# system params
sr=24000
frame_rate=75
device='cpu' #best for inference
buffersize = 320 #[NOTE - not tested on values other than 24000/75 - the frame length of encodec codes in samples]

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>"User" Synth/Encodec Parameters</b>

In [ ]:
g_hopsize=8  # This has a big effect on the off-line generate() because the smaller it is, the more old fashioned context switching time we spend
g_chunksize=24

offline_render_duration=20 #seconds
offline_render_frames=offline_render_duration*75

profiles=["syntex1", "syntex7", "nsynth2", "water", "ddsp_violin", "babble", "nsynth7", "engineC"]
synthprofile=profiles[1] # <<<<<===============================

# if you trained on discrete values, you can force the widget parameters to be discrete, too. See the "nsynth2" profile for pitch. 
# You could also force your "one hot" values to be either 0 or 1 this way, too (though it is more fun to use continuous values for these)
g_desc_param_vals=None # by default - changed in some profiles below


if synthprofile=="syntex1" : 
    g_param_labels = ["Pitch", "Amp"]
    g_norm_param_vals = [.5, .5]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:1] # the first n will be passed to the RNN generator

# elif synthprofile=="syntex7" :
#     #g_param_labels = ["c1", "c2", "c3", "c4", "c5", "c6", "c7", "param 1", "Amp"]
#     g_param_labels = ['Chirps 1H', 'Applause 1H', 'Bugs 1H', 'Peepers 1H', 'Pistons 1H', 'Wind 1H', 'TokWotal 1H', "param 1"]
#     g_norm_param_vals = [1,  0,     0,    0,    0,    0,    0,    0.5]  #for the synth whose paramters can be different than those it sends to the NN
#     g_init_cond=g_norm_param_vals[:8] # the first n will be passed to the RNN generator
#     checkpoint_fname =   "last_checkpoint.pt"
#     #run_directory = str(Path('../output_keep/20250924_144943_multiclass_test'))
#     #run_directory = str(Path('output/20251024_183850_multiclass_test_softcascade'))
#     #run_directory = str(Path('output/20251025_161452_multiclass_test_HARDcascade'))
#     #run_directory = str(Path('output/20251027_181934_multiclass_test_SOFTcascade'))
    
#     run_directory = str(Path('output/20251128_140835_multiclass_test_SOFTcascade'))
#    #run_directory = str(Path('output/20260107_162740_multiclass_test_HARDcascade'))
#     run_directory = str(Path('output/20260121_150526_datapreptest'))
    
#     checkpoint_fname =   "last_checkpoint.pt" #"checkpoint_125.pt" # "checkpoint_25.pt" # "checkpoint_25.pt" #

elif synthprofile=="syntex7" :
    #g_param_labels = ["c1", "c2", "c3", "c4", "c5", "c6", "c7", "param 1", "Amp"]
    g_param_labels = ['param1', 'Chirps 1H', 'Applause 1H', 'Bugs 1H', 'Peepers 1H', 'Pistons 1H', 'Wind 1H', 'TokWotal 1H' ]
    g_norm_param_vals = [0.95 ,  0,     0,    1,    0,    0,    0,    0]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:8] # the first n will be passed to the RNN generator
    checkpoint_fname =   "last_checkpoint.pt"
    #run_directory = str(Path('../output_keep/20250924_144943_multiclass_test'))
    #run_directory = str(Path('output/20251024_183850_multiclass_test_softcascade'))
    #run_directory = str(Path('output/20251025_161452_multiclass_test_HARDcascade'))
    #run_directory = str(Path('output/20251027_181934_multiclass_test_SOFTcascade'))
    
    #run_directory = str(Path('output/20251128_140835_multiclass_test_SOFTcascade'))
    #run_directory = str(Path('output/20260107_162740_multiclass_test_HARDcascade'))
    #run_directory = str(Path('output/20260121_150526_datapreptest'))
    #run_directory = str(Path('output/20260122_134328_datapreptest'))
    #run_directory = str(Path('output/20260203_160208_datapreptest_dynamic_75_images'))
    run_directory = str(Path('output/20260205_183830_datapreptest_dynamic_forkad'))
    checkpoint_fname =   "last_checkpoint.pt" #"checkpoint_125.pt" # "checkpoint_25.pt" # "checkpoint_25.pt" #
    
elif synthprofile=="nsynth2" : #nsynth
    g_param_labels = ["class", "pitch", "amp"]
    g_norm_param_vals = [0,       .5,     .5 ]  # for the synth whose paramters can be different than those it sends to the NN
    g_desc_param_vals = [0,       13,      0 ]  # 0 for float vals, n for n discrete values including endpoints
    g_init_cond=g_norm_param_vals[:]            # for the neural network
    run_directory = str(Path('output/20251024_140730_nsynth_soft_cascade'))
    checkpoint_fname =   "last_checkpoint.pt"
    
elif synthprofile=="water" :
    g_param_labels = ["fill_level"]
    g_norm_param_vals = [.5 ]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network
    # run_directory = str(Path('output/20251024_124629_quickstart_softcascade'))
    # run_directory = str(Path('output/20251120_135045_water_0'))

    

    run_directory = str(Path("../quickstart/output_26.02.03.15.49")) # Path to save the trained model and configs
    checkpoint_fname = "last_checkpoint.pt"   # Checkpoint to use for inference (e.g., "last_checkpoint.pt" or "checkpoint_75.pt")

    

elif synthprofile=="ddsp_violin" :
    g_param_labels = ["pitch", "amp"]
    g_norm_param_vals = [.5, .7 ]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network
    run_directory = str(Path('output/20251017_113920_ddsp_violin_seql_75_corrected'))
    checkpoint_fname =   "last_checkpoint.pt" # "checkpoint_25.pt" # "checkpoint_25.pt" #


elif synthprofile=="babble" :
    g_param_labels = ["sex"]
    g_norm_param_vals = [.5]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network
    run_directory = str(Path('output/20251119_163720_babble_gender_mf'))
    checkpoint_fname =   "checkpoint_425.pt" # "last_checkpoint.pt" # "checkpoint_25.pt" # 


elif synthprofile=="nsynth7" :
    g_param_labels = ["pitch", "velocity", "003clarinet", "006frenchhorn","094bass", "162flute","272voice-ee","295voice-oo","887organ"]
    g_norm_param_vals = [.5,.4,1,0,0,0,0,0,0]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network
    run_directory = str(Path('output/20251128_111405_nsynth7_trial2'))
    checkpoint_fname =   "checkpoint_175.pt" # "last_checkpoint.pt" # "checkpoint_425.pt" #  "checkpoint_25.pt" # 

elif synthprofile=="engineC" :
    g_param_labels = ["RPM", "Torque"]
    g_norm_param_vals = [.5,.5]  #for the synth whose paramters can be different than those it sends to the NN
    g_init_cond=g_norm_param_vals[:]             # for the neural network
    run_directory = str(Path('output/20260109_175027_engineC_trial'))
    run_directory = str(Path('output/20260124_161336_engineC_SOFTtrial_TFCYCLE_TAU.35_topnsoft_postmerge'))
    checkpoint_fname =   "checkpoint_75.pt" # "last_checkpoint.pt" # "checkpoint_175.pt" #  "checkpoint_425.pt" #  "checkpoint_25.pt" # 

    

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>RNN Parameters</b>

In [ ]:
#Used to alter the config after it is loaded from the checkpoint
g_top_n = 8 #'Sample from the top N most likely outputs.'
g_temperature = .9 #'Controls the randomness of predictions.'
g_sample_mode="sample"

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>loading</b>

In [ ]:
# load the encoder model 
#####################################################################
enc_model = EncodecModel.from_pretrained("facebook/encodec_24khz")
enc_model.eval()
enc_model.device

# load the RNN model 
#####################################################################
config_path = os.path.join(run_directory, "config_v2.pt")
checkpoint_path = os.path.join(run_directory, "checkpoints", checkpoint_fname) 

assert os.path.exists(run_directory), f"Run directory not found: {run_directory}"
assert os.path.exists(config_path), f"Config file not found: {config_path}"
assert os.path.exists(checkpoint_path), f"Checkpoint file not found: {checkpoint_path}"

saved_configs = torch.load(config_path, weights_only=False)
model_config = GRUModelConfig(**saved_configs["model_config"])
data_config = LatentDatasetConfig(**saved_configs["data_config"])

# could be different from config defaults or what we trained with
# model_config.cascade='hard' # 
# model_config.hard_sample_mode=g_sample_mode
# model_config.top_n_hard=g_top_n
# model_config.temperature_hard=g_temperature

rnngen = RNNGeneratorSoft.from_checkpoint(checkpoint_path, model_config, data_config, enc_model, g_chunksize, g_hopsize, sample_mode_outside=g_sample_mode, top_k_outside=g_top_n, temperature_outside =g_temperature)
print("Model successfully loaded from checkpoint.")
print(f"Using device = {device}") 

In [ ]:
model_config

In [ ]:
data_config

In [ ]:
# # Visualize hops shiting into the "right" side of chunks
# print(f"rnngen.codebuf is {rnngen.codebuf}")
# newfoo = rnngen.getNextCodeChunk(g_init_cond, g_hopsize)
# print(f"rnngen.codebuf is {rnngen.codebuf}")

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>ON LINE, Realtime Usage</b>

In [ ]:
GENS = {
    "EncodecRTPlayer": lambda: EncodecRTPlayer(rnngen, sr, frame_rate, buffersize, g_chunksize,  g_hopsize, g_norm_param_vals, g_param_labels),
    "Sine": SineGenerator,            # defined in the default system
    "Noisy LP": NoisyLPGenerator,     # defined in the default system
}
print(f"Will use samplerate = {sr} and blocksize = {buffersize}")

print(f"g_norm_param_vals = {g_norm_param_vals}")
# most external synths don't support 24kHz, so we wll resample the encodec output to 48K
synth, ui = build_synth_ui(GENS, samplerate=48000, blocksize=buffersize*2, channels=1)


<div style="width: 20%; height: 3px; background-color: blue;"></div>
<b>Since we can not print from a different thread while running RT, some messages are recorded in attributes for viewing after stopping the sound</b>

In [ ]:
print(getattr(synth.gen, "_last_error", None))  #will show "missed swaps" if you hopsize is too small to keep up with next hop requests

In [ ]:
print(getattr(synth.gen, "_callrecord", None))

In [ ]:
print(getattr(synth.gen, "_decodetime", None))

In [ ]:
print(getattr(synth.gen, "_setnormval", None))

<div style="width: 100%; height: 20px; background-color: maroon;"></div>
<b>EncodecRTPlayer synthesizer OFF LINE TEST 1 (not meant for users)</b>

In [ ]:
foo=EncodecRTPlayer(rnngen, sr, frame_rate, buffersize,  g_chunksize, g_hopsize, g_norm_param_vals, g_param_labels)
c=[]
hops=int(offline_render_duration*frame_rate/g_hopsize)

for hopnum in range(0,hops) :
    c=  np.concatenate((c , foo.thisaudioseq), axis=0) 
    foo.thisaudioseq=foo.getNextAudioHop()
print(f"time spent decoding hops={hops}, {offline_render_duration}  secs of audio = {foo._decodetime:.2f}")


In [ ]:
display(Audio(c, rate=sr))

In [ ]:
#print(f"{foo._callrecord}")

<div style="width: 100%; height: 20px; background-color: maroon;"></div>
<b>EncodecRTPlayer OFF LINE TEST 2 - calling generate()  ("simulating" the os system portaudio calls into the synth)</b>

In [ ]:
bar= EncodecRTPlayer(rnngen, sr, frame_rate, buffersize, g_chunksize,  g_hopsize, g_norm_param_vals, g_param_labels)

In [ ]:
#you can't just call generate in a loop like this because the calls happen way to fast for the NN to meet the needs of the fill buffer callback
# for buffernum in range(0,1375) :
#     audio_out =  np.concatenate((audio_out, bar.generate(buffersize, sr)), axis=0) 
    

In [ ]:
#This function uses generate (but waits for the "next hop" generation from the neural net so it doesn't lose any audio.
# You can think of this as rendering as fast as it can - faster than real time, but only as fast as the NN can produce the data
#
def render_offline(player, n_blocks, sr, buffersize):
    blocks = []
    for i in range(n_blocks):
        # if we’re at the hop boundary, ensure next hop is ready
        if player.currentchunkframe == 0:
            # make sure a future is scheduled
            if player._next_future is None:
                player._schedule_next_hop()
            # wait here until it finishes, then collect
            if player._next_future is not None:
                player._next_future.result()   # blocks here offline
                player._try_collect_next()
        blocks.append(player.generate(buffersize, sr).copy())
    return np.concatenate(blocks).astype(np.float32)

# usage:
#audio_out = render_offline(bar, 1375, sr, buffersize)

In [ ]:

start_time = time.monotonic()
audio_out = render_offline(bar, offline_render_frames, sr, buffersize)
rnnelapsed_time = time.monotonic() - start_time
print(f"RNN time to generate  offline_render_frames = {offline_render_frames}, {offline_render_duration} secs of sound: {rnnelapsed_time:.2f}. Converting to audio...")


In [ ]:
display(Audio(audio_out, rate=sr)) 

In [ ]:
g_norm_param_vals

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b style="font-size: 20px;">RNNGenerator testing -  the intended way to generate "off line" </b>

In [ ]:
#++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
#  NOW with RNNGenerator Directly 
#++++++++++++++++++++++++++++++++++++++++++++++++++
bazrnn=RNNGeneratorSoft.from_checkpoint(checkpoint_path, model_config, data_config, enc_model, g_chunksize, g_hopsize, sample_mode_outside=g_sample_mode, top_k_outside=g_top_n, temperature_outside =g_temperature)
#warm up prevents noise bursts at the begininng of the audio run
_ = bazrnn.warmup(g_init_cond, 10)

# Note that the param array is used for the whole hop. 
# Update the param array per hop for dynamic control 

segments = []
numhops=int(offline_render_frames/g_hopsize)
print(f'numhops will be {numhops}')
#params = np.linspace(0, 1, numhops, dtype=np.float32)[:, np.newaxis]


start = time.perf_counter()
for i in range(numhops):
    p = torch.tensor([g_init_cond], dtype=torch.float32, device=device)
    params_seq = p.view(1, -1).expand(g_hopsize, -1)  # shape (T, 2), view with stride 0
    a=bazrnn.getNextAudioHop(params_seq)  # returns a 1D float32 array like in your example
    segments.append(a)

fullaudio = np.concatenate(segments, dtype=np.float32)  # one allocation

elapsed = time.perf_counter() - start

print(f'Total time to render {fullaudio.shape[0]/sr} secs: {elapsed}')
print(f"Generated { fullaudio.shape[0]} samples at {sr} Hz in {elapsed:.2f}s.")


In [ ]:
display(Audio(fullaudio, rate=sr)) 

In [ ]:
#if you call with hopsizes larger than the chunk you set up, you'll get them, but it won't take advantage of the chunking strategy)

In [ ]:
g_init_cond

In [ ]:
#foo = bazrnn.warmup([.5,.5, .5], 1000)  
#foo=bazrnn.warmup([0, 0, 0, 0, 0, 0, 1, 0.5], 1000)
foo=bazrnn.warmup(g_init_cond, 10)
display(Audio(foo, rate=sr)) 

In [ ]:
params_seq